# Lab 02 Data Acquisition for NLP



## 0 Setup and Input Files

The following cell installs only packages that are missing from the current notebook environment.


In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "pdfplumber": "pdfplumber",
    "docx": "python-docx",
    "pandas": "pandas",
    "requests": "requests",
    "nltk": "nltk",
    "reportlab": "reportlab",
}

missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )
else:
    print("All required packages are already available.")


All required packages are already available.


In [2]:
import csv
import math
import re
import textwrap
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint

import nltk
import pandas as pd
import pdfplumber
import requests
from docx import Document
from docx.oxml.ns import qn
from docx.text.paragraph import Paragraph
from nltk.corpus import stopwords
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

FALLBACK_ENGLISH_STOPWORDS = set(
    """
    a about above after again against all am an and any are aren't as at be
    because been before being below between both but by can can't cannot could
    couldn't did didn't do does doesn't doing don't down during each few for
    from further had hadn't has hasn't have haven't having he he'd he'll he's
    her here here's hers herself him himself his how how's i i'd i'll i'm i've
    if in into is isn't it it's its itself just me more most mustn't my myself
    no nor not of off on once only or other ought our ours ourselves out over
    own same shan't she she'd she'll she's should shouldn't so some such than
    that that's the their theirs them themselves then there there's these they
    they'd they'll they're they've this those through to too under until up
    very was wasn't we we'd we'll we're we've were weren't what what's when
    when's where where's which while who who's whom why why's with won't would
    wouldn't you you'd you'll you're you've your yours yourself yourselves
    """.split()
)

try:
    stopwords.words("english")
except LookupError:
    print("Downloading the NLTK English stopword corpus...")
    download_succeeded = nltk.download("stopwords", quiet=True)
    if not download_succeeded:
        print("Download unavailable; the notebook will use its embedded English fallback list.")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)
print("Imports completed.")


Imports completed.


### Configuration

- Keep `USE_DEMO_FILES = True` to generate sample inputs automatically.
- Set it to `False` and replace the three custom paths to analyze your own files.
- Keep `USE_DEMO_INPUTS = True` for automatic execution. Set it to `False` to enter the keyword and phrase through `input()` as required in Tasks 1(d) and 2(c).


In [3]:
USE_DEMO_FILES = True
USE_DEMO_INPUTS = True

DEMO_KEYWORD = "language"
DEMO_SEARCH_PHRASE = "natural language"

WORK_DIR = Path("lab02_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# These paths are used only when USE_DEMO_FILES is False.
CUSTOM_PDF_PATH = Path("your_multi_page_file.pdf")
CUSTOM_DOCX_A_PATH = Path("your_document_a.docx")
CUSTOM_DOCX_B_PATH = Path("your_document_b.docx")

print("Working directory:", WORK_DIR)


Working directory: lab02_work


In [4]:
def create_demo_pdf(path: Path) -> None:
    page_texts = [
        (
            "Natural Language Processing Overview. Natural language processing helps computers "
            "understand written and spoken human communication. Modern systems analyze words, "
            "sentences, and context to discover useful patterns. Visit https://example.com/nlp "
            "for course notes published in 2026. Students practice extraction, cleaning, "
            "tokenization, and frequency analysis during this laboratory exercise."
        ),
        (
            "PDF Text Extraction. A multi-page PDF stores content on separate pages, so every "
            "page should be processed individually before the results are combined. Reliable "
            "code handles empty pages without crashing. The extracted strings can support "
            "search, statistics, and later natural language processing experiments. This page "
            "contains 5 short examples and several punctuation marks!"
        ),
        (
            "Text Preprocessing for NLP. Cleaning removes URLs, digits, and unwanted symbols "
            "while preserving meaningful sentence boundaries. Tokenization converts cleaned "
            "sentences into words. Stopword removal reduces very common words, allowing important "
            "terms such as language, document, model, and analysis to become more visible. "
            "See www.example.org/preprocessing for 10 additional examples."
        ),
        (
            "Document Metadata and Tables. Page-level metadata includes the page number, raw text, "
            "word count, and character count. DOCX files also store paragraph styles and tables. "
            "Structured extraction makes it possible to compare headings, body paragraphs, and "
            "table cells. Careful validation prevents silent errors in downstream analysis."
        ),
        (
            "Web APIs and Nested JSON. Public endpoints return users, posts, comments, and tasks "
            "as JSON records. Joining related identifiers creates useful summaries for each user. "
            "Nested address and company fields can be flattened into a tabular dataset. Completion "
            "rates, geographic coordinates, and comment counts provide practical examples of data "
            "acquisition for natural language projects."
        ),
    ]

    pdf = canvas.Canvas(str(path), pagesize=letter)
    width, height = letter
    for page_number, content in enumerate(page_texts, start=1):
        pdf.setFont("Helvetica-Bold", 15)
        pdf.drawString(54, height - 55, f"Lab 02 Sample Document - Page {page_number}")
        pdf.setFont("Helvetica", 11)
        text_object = pdf.beginText(54, height - 90)
        text_object.setLeading(17)
        for line in textwrap.wrap(content, width=88):
            text_object.textLine(line)
        pdf.drawText(text_object)
        pdf.showPage()
    pdf.save()


def create_demo_docx_files(path_a: Path, path_b: Path) -> None:
    shared_paragraph = (
        "Data acquisition is the first stage of a reliable natural language processing pipeline."
    )

    doc_a = Document()
    doc_a.add_paragraph("Document A NLP Notes", style="Title")
    doc_a.add_heading("Natural Language Processing", level=1)
    doc_a.add_paragraph(
        "Natural language processing enables computers to analyze human language in useful and repeatable ways."
    )
    doc_a.add_paragraph(shared_paragraph)
    doc_a.add_heading("Tokenization", level=2)
    doc_a.add_paragraph(
        "Tokenization divides a document into smaller units such as sentences, words, or subword pieces."
    )
    doc_a.add_heading("Data Collection", level=2)
    doc_a.add_paragraph(
        "High quality documents should be collected from relevant, lawful, and well documented sources."
    )
    table = doc_a.add_table(rows=1, cols=3)
    table.style = "Table Grid"
    headers = ["Student", "Course", "Score"]
    for index, value in enumerate(headers):
        table.rows[0].cells[index].text = value
    for row_values in [
        ["Ali", "NLP", "88"],
        ["Sara", "AI", ""],
        ["Hamza", "NLP", "91"],
    ]:
        cells = table.add_row().cells
        for index, value in enumerate(row_values):
            cells[index].text = value
    doc_a.add_paragraph("This paragraph appears after the first table.")
    doc_a.save(path_a)

    doc_b = Document()
    doc_b.add_paragraph("Document B Text Analytics Notes", style="Title")
    doc_b.add_heading("Natural Language Processing", level=1)
    doc_b.add_paragraph(
        "Natural language processing supports search, summarization, classification, and question answering systems."
    )
    doc_b.add_paragraph("  " + shared_paragraph.upper() + "  ")
    doc_b.add_heading("Text Cleaning", level=2)
    doc_b.add_paragraph(
        "Text cleaning removes noise while retaining the linguistic information needed for later analysis."
    )
    doc_b.add_heading("Data Collection", level=2)
    doc_b.add_paragraph(
        "Ethical data collection records the source, permission, quality, and limitations of every dataset."
    )
    doc_b.save(path_b)


if USE_DEMO_FILES:
    PDF_PATH = WORK_DIR / "sample_multi_page.pdf"
    DOCX_A_PATH = WORK_DIR / "document_a.docx"
    DOCX_B_PATH = WORK_DIR / "document_b.docx"
    create_demo_pdf(PDF_PATH)
    create_demo_docx_files(DOCX_A_PATH, DOCX_B_PATH)
else:
    PDF_PATH = CUSTOM_PDF_PATH
    DOCX_A_PATH = CUSTOM_DOCX_A_PATH
    DOCX_B_PATH = CUSTOM_DOCX_B_PATH

for label, path in {
    "PDF": PDF_PATH,
    "Document A": DOCX_A_PATH,
    "Document B": DOCX_B_PATH,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"{label} was not found: {path}")
    print(f"{label}: {path}")


PDF: lab02_work/sample_multi_page.pdf
Document A: lab02_work/document_a.docx
Document B: lab02_work/document_b.docx


## Task 1 Multi-Page PDF Extraction with Word Frequency Analysis

### 1(a) Extract every PDF page separately


In [5]:
with pdfplumber.open(PDF_PATH) as pdf:
    page_texts = [(page.extract_text() or "") for page in pdf.pages]

print("Total pages extracted:", len(page_texts))


Total pages extracted: 5


### 1(b) Merge, clean, lowercase, and tokenize the page text


In [6]:
merged_text = "\n".join(page_texts)
cleaned_for_tokens = re.sub(r"[^A-Za-z\s]", " ", merged_text)
tokens = cleaned_for_tokens.lower().split()

print("Total token count:", len(tokens))


Total token count: 272


### 1(c) Build a word-frequency dictionary and remove common stopwords


In [7]:
common_stopwords = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "has", "he", "in", "is", "it", "its", "of", "on", "or", "that",
    "the", "to", "was", "were", "will", "with", "this", "these", "those",
    "their", "they", "them", "into", "can", "should", "such", "while",
}

filtered_tokens = [token for token in tokens if token not in common_stopwords]
word_frequency = dict(Counter(filtered_tokens))
top_10_words = sorted(
    word_frequency.items(), key=lambda item: (-item[1], item[0])
)[:10]

print("Top 10 most frequent words:")
for word, count in top_10_words:
    print(f"{word:<20} {count}")


Top 10 most frequent words:
page                 10
document             7
lab                  5
language             5
sample               5
natural              4
analysis             3
examples             3
extraction           3
processing           3


### 1(d) Find every sentence containing a user-supplied keyword


In [8]:
keyword = (
    DEMO_KEYWORD
    if USE_DEMO_INPUTS
    else input("Enter a keyword to search for: ").strip()
)

matches_found = False
for page_number, page_text in enumerate(page_texts, start=1):
    normalized_page = re.sub(r"\s+", " ", page_text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", normalized_page)
    for sentence in sentences:
        if keyword.lower() in sentence.lower():
            matches_found = True
            print(f"Page {page_number}: {sentence.strip()}")

if not matches_found:
    print(f'No sentence containing "{keyword}" was found.')


Page 1: Lab 02 Sample Document - Page 1 Natural Language Processing Overview.
Page 1: Natural language processing helps computers understand written and spoken human communication.
Page 2: The extracted strings can support search, statistics, and later natural language processing experiments.
Page 3: Stopword removal reduces very common words, allowing important terms such as language, document, model, and analysis to become more visible.
Page 5: Completion rates, geographic coordinates, and comment counts provide practical examples of data acquisition for natural language projects.


## Task 2 Structured Page Metadata Extraction and CSV Export

### 2(a) Build `pages_data` using exactly the required keys


In [9]:
if len(page_texts) < 5:
    raise ValueError("Task 2 requires a PDF containing at least 5 pages.")

pages_data = []
for page_number, raw_text in enumerate(page_texts, start=1):
    pages_data.append(
        {
            "page_number": page_number,
            "raw_text": raw_text,
            "word_count": len(raw_text.split()),
            "char_count": sum(1 for character in raw_text if not character.isspace()),
        }
    )

print("First 2 page records:")
pprint(pages_data[:2])


First 2 page records:
[{'char_count': 370,
  'page_number': 1,
  'raw_text': 'Lab 02 Sample Document - Page 1\n'
              'Natural Language Processing Overview. Natural language '
              'processing helps computers\n'
              'understand written and spoken human communication. Modern '
              'systems analyze words,\n'
              'sentences, and context to discover useful patterns. Visit '
              'https://example.com/nlp for\n'
              'course notes published in 2026. Students practice extraction, '
              'cleaning, tokenization,\n'
              'and frequency analysis during this laboratory exercise.',
  'word_count': 53},
 {'char_count': 348,
  'page_number': 2,
  'raw_text': 'Lab 02 Sample Document - Page 2\n'
              'PDF Text Extraction. A multi-page PDF stores content on '
              'separate pages, so every page\n'
              'should be processed individually before the results are '
              'combined. Reliable

### 2(b) Find maximum, minimum non-zero, and average page word counts


In [10]:
highest_page = max(pages_data, key=lambda page: page["word_count"])
non_zero_pages = [page for page in pages_data if page["word_count"] > 0]
average_word_count = round(
    sum(page["word_count"] for page in pages_data) / len(pages_data), 2
)

print("Page with the highest word count:", highest_page["page_number"])
print("Word count:", highest_page["word_count"])
print("Full content:")
print(highest_page["raw_text"])

if non_zero_pages:
    lowest_non_zero_page = min(non_zero_pages, key=lambda page: page["word_count"])
    print("\nPage with the lowest non-zero word count:", lowest_non_zero_page["page_number"])
    print("Word count:", lowest_non_zero_page["word_count"])
else:
    lowest_non_zero_page = None
    print("\nNo page contains any words.")

print("Average word count across all pages:", average_word_count)


Page with the highest word count: 5
Word count: 61
Full content:
Lab 02 Sample Document - Page 5
Web APIs and Nested JSON. Public endpoints return users, posts, comments, and tasks as
JSON records. Joining related identifiers creates useful summaries for each user. Nested
address and company fields can be flattened into a tabular dataset. Completion rates,
geographic coordinates, and comment counts provide practical examples of data
acquisition for natural language projects.

Page with the lowest non-zero word count: 1
Word count: 53
Average word count across all pages: 56.4


### 2(c) Search for a phrase and handle `PhraseNotFoundError` gracefully


In [11]:
class PhraseNotFoundError(Exception):
    """Raised when a search phrase is absent from every PDF page."""


search_phrase = (
    DEMO_SEARCH_PHRASE
    if USE_DEMO_INPUTS
    else input("Enter a phrase to search for: ").strip()
)

try:
    phrase_matches = []
    for page in pages_data:
        occurrence_count = page["raw_text"].lower().count(search_phrase.lower())
        if occurrence_count > 0:
            phrase_matches.append((page["page_number"], occurrence_count))

    if not phrase_matches:
        raise PhraseNotFoundError(
            f'The phrase "{search_phrase}" was not found in the PDF.'
        )

    for page_number, occurrence_count in phrase_matches:
        print(
            f"Page {page_number}: {occurrence_count} occurrence(s) of "
            f'"{search_phrase}"'
        )
except PhraseNotFoundError as error:
    print("Search completed:", error)


Page 1: 2 occurrence(s) of "natural language"
Page 2: 1 occurrence(s) of "natural language"
Page 5: 1 occurrence(s) of "natural language"


### 2(d) Export the page summary with the built-in `csv` module


In [12]:
PDF_SUMMARY_PATH = WORK_DIR / "pdf_summary.csv"
fieldnames = ["page_number", "word_count", "char_count"]

with PDF_SUMMARY_PATH.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    for page in pages_data:
        writer.writerow({field: page[field] for field in fieldnames})

print("CSV contents:")
with PDF_SUMMARY_PATH.open("r", newline="", encoding="utf-8") as csv_file:
    for row in csv.reader(csv_file):
        print(row)


CSV contents:
['page_number', 'word_count', 'char_count']
['1', '53', '370']
['2', '60', '348']
['3', '55', '361']
['4', '53', '310']
['5', '61', '354']


## Task 3 PDF Text Preprocessing Pipeline for NLP

### 3(a) Apply the four required `re.sub()` cleaning stages


In [13]:
all_pdf_text = "\n".join(page_texts)
character_count_before = len(all_pdf_text)

cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", all_pdf_text)
cleaned_text = re.sub(r"\d+", " ", cleaned_text)
cleaned_text = re.sub(r"[^A-Za-z. ]+", " ", cleaned_text)
cleaned_text = re.sub(r" +", " ", cleaned_text).strip()

character_count_after = len(cleaned_text)
print("Character count before cleaning:", character_count_before)
print("Character count after cleaning:", character_count_after)


Character count before cleaning: 2024
Character count after cleaning: 1901


### 3(b) Lowercase and split the cleaned text into sentences


In [14]:
lowercase_text = cleaned_text.lower()
raw_sentence_parts = re.split(r"[.!?]", lowercase_text)
stripped_sentence_parts = [sentence.strip() for sentence in raw_sentence_parts]
sentences = [sentence for sentence in stripped_sentence_parts if sentence]

print("Sentence count before removing empty strings:", len(stripped_sentence_parts))
print("Sentence count after removing empty strings:", len(sentences))


Sentence count before removing empty strings: 25
Sentence count after removing empty strings: 24


### 3(c) Tokenize, remove NLTK stopwords, and filter short sentences


In [15]:
try:
    nltk_stopwords = set(stopwords.words("english"))
except LookupError:
    nltk_stopwords = FALLBACK_ENGLISH_STOPWORDS
tokenized_sentences = []

for sentence in sentences:
    sentence_tokens = re.findall(r"[a-z]+", sentence)
    remaining_tokens = [
        token for token in sentence_tokens if token not in nltk_stopwords
    ]
    tokenized_sentences.append(remaining_tokens)

nlp_ready_sentences = [
    " ".join(sentence_tokens)
    for sentence_tokens in tokenized_sentences
    if len(sentence_tokens) >= 5
]

print("Sentence count before the 5-token filter:", len(tokenized_sentences))
print("Sentence count after the 5-token filter:", len(nlp_ready_sentences))
print("\nThree sample processed sentences:")
for sample_number, sentence in enumerate(nlp_ready_sentences[:3], start=1):
    print(f"{sample_number}. {sentence}")


Sentence count before the 5-token filter: 24
Sentence count after the 5-token filter: 22

Three sample processed sentences:
1. lab sample document page natural language processing overview
2. natural language processing helps computers understand written spoken human communication
3. modern systems analyze words sentences context discover useful patterns


### 3(d) Save and inspect the NLP-ready sentences


In [16]:
NLP_READY_PATH = WORK_DIR / "nlp_ready_output.txt"

with NLP_READY_PATH.open("w", encoding="utf-8") as output_file:
    for sentence in nlp_ready_sentences:
        output_file.write(sentence + "\n")

with NLP_READY_PATH.open("r", encoding="utf-8") as input_file:
    saved_sentences = [line.rstrip("\n") for line in input_file]

print("Total lines written:", len(saved_sentences))
if saved_sentences:
    longest_sentence = max(saved_sentences, key=len)
    shortest_sentence = min(saved_sentences, key=len)
    print("Longest sentence by character length:", longest_sentence)
    print("Shortest sentence by character length:", shortest_sentence)
else:
    print("No sentence met the minimum-token requirement.")


Total lines written: 22
Longest sentence by character length: completion rates geographic coordinates comment counts provide practical examples data acquisition natural language projects
Shortest sentence by character length: lab sample document page pdf text extraction


## Task 4 DOCX Paragraph Classification and Statistical Analysis

### 4(a) Extract paragraph text and style names


In [17]:
doc_a = Document(DOCX_A_PATH)
paragraph_records = [
    {"text": paragraph.text, "style": paragraph.style.name}
    for paragraph in doc_a.paragraphs
]
unique_style_names = sorted({record["style"] for record in paragraph_records})

print("Total paragraph count:", len(paragraph_records))
print("Unique style names:", unique_style_names)


Total paragraph count: 9
Unique style names: ['Heading 1', 'Heading 2', 'Normal', 'Title']


### 4(b) Classify paragraphs into exactly three groups


In [18]:
paragraph_groups = {"Headings": [], "Body": [], "Other/Empty": []}

for record in paragraph_records:
    text = record["text"]
    style_name = record["style"]

    if not text.strip():
        group_name = "Other/Empty"
    elif style_name.startswith("Heading"):
        group_name = "Headings"
    elif style_name == "Normal" or "Body" in style_name:
        group_name = "Body"
    else:
        group_name = "Other/Empty"

    paragraph_groups[group_name].append(record)

for group_name, records in paragraph_groups.items():
    print(f"{group_name}: {len(records)}")


Headings: 3
Body: 5
Other/Empty: 1


### 4(c) Compute statistics for qualifying Body paragraphs


In [19]:
qualifying_body_paragraphs = []
for record in paragraph_groups["Body"]:
    word_count = len(record["text"].split())
    if word_count > 5:
        qualifying_body_paragraphs.append(
            {
                "text": record["text"],
                "word_count": word_count,
                "char_count": sum(
                    1 for character in record["text"] if not character.isspace()
                ),
            }
        )

if qualifying_body_paragraphs:
    body_with_most_words = max(
        qualifying_body_paragraphs, key=lambda paragraph: paragraph["word_count"]
    )
    body_with_fewest_words = min(
        qualifying_body_paragraphs, key=lambda paragraph: paragraph["word_count"]
    )
    mean_body_word_count = round(
        sum(paragraph["word_count"] for paragraph in qualifying_body_paragraphs)
        / len(qualifying_body_paragraphs),
        2,
    )

    print("Body paragraph with the highest word count:")
    pprint(body_with_most_words)
    print("\nBody paragraph with the lowest word count above 5:")
    pprint(body_with_fewest_words)
    print("\nMean word count:", mean_body_word_count)
else:
    print("No Body paragraph contains more than 5 words.")


Body paragraph with the highest word count:
{'char_count': 89,
 'text': 'Natural language processing enables computers to analyze human '
         'language in useful and repeatable ways.',
 'word_count': 14}

Body paragraph with the lowest word count above 5:
{'char_count': 39,
 'text': 'This paragraph appears after the first table.',
 'word_count': 7}

Mean word count: 12.2


### 4(d) Build, sort, and pretty-print the nested style dictionary


In [20]:
style_summary = {}
for record in paragraph_records:
    style_name = record["style"]
    style_summary.setdefault(style_name, {"count": 0, "texts": []})
    style_summary[style_name]["count"] += 1
    style_summary[style_name]["texts"].append(record["text"])

sorted_style_summary = dict(
    sorted(
        style_summary.items(),
        key=lambda item: (-item[1]["count"], item[0]),
    )
)

display_style_summary = {
    style_name: {
        "count": details["count"],
        "texts": details["texts"][:2]
        if details["count"] > 2
        else details["texts"],
    }
    for style_name, details in sorted_style_summary.items()
}

pprint(display_style_summary)


{'Heading 1': {'count': 1, 'texts': ['Natural Language Processing']},
 'Heading 2': {'count': 2, 'texts': ['Tokenization', 'Data Collection']},
 'Normal': {'count': 5,
            'texts': ['Natural language processing enables computers to '
                      'analyze human language in useful and repeatable ways.',
                      'Data acquisition is the first stage of a reliable '
                      'natural language processing pipeline.']},
 'Title': {'count': 1, 'texts': ['Document A NLP Notes']}}


## Task 5 DOCX Table Extraction and Pandas Integration

### 5(a) Extract each table as a list of lists


In [21]:
doc_with_table = Document(DOCX_A_PATH)
all_tables_data = []

for table_number, table in enumerate(doc_with_table.tables, start=1):
    table_data = [
        [cell.text for cell in row.cells]
        for row in table.rows
    ]
    all_tables_data.append(table_data)
    row_count = len(table_data)
    column_count = len(table_data[0]) if table_data else 0
    print(f"Table {table_number} dimensions: {row_count} x {column_count}")

print("Total number of tables found:", len(all_tables_data))


Table 1 dimensions: 4 x 3
Total number of tables found: 1


### 5(b) Convert the first table to a DataFrame and count empty cells


In [22]:
if not all_tables_data:
    raise ValueError("The selected DOCX file does not contain a table.")

first_table_data = all_tables_data[0]
table_header = first_table_data[0]
table_rows = first_table_data[1:]
first_table_df = pd.DataFrame(table_rows, columns=table_header)

print("First table as a DataFrame:")
print(first_table_df.to_string(index=False))

empty_or_none_per_column = (
    first_table_df.isna() | first_table_df.apply(lambda column: column.str.strip().eq(""))
).sum()
print("\nEmpty or None cells per column:")
print(empty_or_none_per_column.to_string())


First table as a DataFrame:
Student Course Score
    Ali    NLP    88
   Sara     AI      
  Hamza    NLP    91

Empty or None cells per column:
Student    0
Course     0
Score      1


### 5(c) Extract paragraphs before the first `w:tbl` element


In [23]:
pre_table_paragraphs = []

for child in doc_with_table.element.body.iterchildren():
    if child.tag == qn("w:tbl"):
        break
    if child.tag == qn("w:p"):
        paragraph = Paragraph(child, doc_with_table)
        pre_table_paragraphs.append(paragraph.text)

print("Number of paragraphs before the first table:", len(pre_table_paragraphs))
for index, paragraph_text in enumerate(pre_table_paragraphs, start=1):
    print(f"{index}. {paragraph_text}")


Number of paragraphs before the first table: 8
1. Document A NLP Notes
2. Natural Language Processing
3. Natural language processing enables computers to analyze human language in useful and repeatable ways.
4. Data acquisition is the first stage of a reliable natural language processing pipeline.
5. Tokenization
6. Tokenization divides a document into smaller units such as sentences, words, or subword pieces.
7. Data Collection
8. High quality documents should be collected from relevant, lawful, and well documented sources.


### 5(d) Save pre-table paragraphs and table data in one text file


In [24]:
DOCX_EXTRACTED_PATH = WORK_DIR / "docx_extracted.txt"

with DOCX_EXTRACTED_PATH.open("w", encoding="utf-8") as output_file:
    output_file.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for paragraph_text in pre_table_paragraphs:
        output_file.write(paragraph_text + "\n")

    output_file.write("\n--- TABLE DATA ---\n")
    for row in first_table_data:
        output_file.write(" | ".join(row) + "\n")

print(DOCX_EXTRACTED_PATH.read_text(encoding="utf-8"))


--- PRE-TABLE PARAGRAPHS ---
Document A NLP Notes
Natural Language Processing
Natural language processing enables computers to analyze human language in useful and repeatable ways.
Data acquisition is the first stage of a reliable natural language processing pipeline.
Tokenization
Tokenization divides a document into smaller units such as sentences, words, or subword pieces.
Data Collection
High quality documents should be collected from relevant, lawful, and well documented sources.

--- TABLE DATA ---
Student | Course | Score
Ali | NLP | 88
Sara | AI | 
Hamza | NLP | 91



## Task 6 Multi-Document Comparison Deduplication and Frequency Report

### 6(a) Compare headings using set operations


In [25]:
document_a = Document(DOCX_A_PATH)
document_b = Document(DOCX_B_PATH)

headings_a = [
    paragraph.text.strip()
    for paragraph in document_a.paragraphs
    if paragraph.style.name.startswith("Heading")
]
headings_b = [
    paragraph.text.strip()
    for paragraph in document_b.paragraphs
    if paragraph.style.name.startswith("Heading")
]

heading_set_a = set(headings_a)
heading_set_b = set(headings_b)

print("Headings common to both documents:", sorted(heading_set_a & heading_set_b))
print("Headings only in Document A:", sorted(heading_set_a - heading_set_b))
print("Headings only in Document B:", sorted(heading_set_b - heading_set_a))


Headings common to both documents: ['Data Collection', 'Natural Language Processing']
Headings only in Document A: ['Tokenization']
Headings only in Document B: ['Text Cleaning']


### 6(b) Merge and deduplicate Normal-style paragraphs


In [26]:
normal_paragraphs_a = [
    paragraph.text
    for paragraph in document_a.paragraphs
    if paragraph.style.name == "Normal"
]
normal_paragraphs_b = [
    paragraph.text
    for paragraph in document_b.paragraphs
    if paragraph.style.name == "Normal"
]

combined_normal_paragraphs = normal_paragraphs_a + normal_paragraphs_b
deduplicated_paragraphs = []
seen_normalized_paragraphs = set()

for paragraph_text in combined_normal_paragraphs:
    normalized_text = paragraph_text.strip().lower()
    if normalized_text not in seen_normalized_paragraphs:
        seen_normalized_paragraphs.add(normalized_text)
        deduplicated_paragraphs.append(paragraph_text.strip())

print("Paragraph count before deduplication:", len(combined_normal_paragraphs))
print("Paragraph count after deduplication:", len(deduplicated_paragraphs))


Paragraph count before deduplication: 9
Paragraph count after deduplication: 8


### 6(c) Build the combined frequency report after stopword removal


In [27]:
comparison_stopwords = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "has", "in", "is", "it", "of", "on", "or", "that", "the", "to",
    "was", "were", "will", "with", "this", "these", "their", "into",
}

paragraph_tokens = []
combined_tokens_without_stopwords = []

for paragraph_text in deduplicated_paragraphs:
    cleaned_paragraph = re.sub(r"[^A-Za-z\s]", " ", paragraph_text.lower())
    current_tokens = cleaned_paragraph.split()
    paragraph_tokens.append(current_tokens)
    combined_tokens_without_stopwords.extend(
        token for token in current_tokens if token not in comparison_stopwords
    )

combined_word_frequency = Counter(combined_tokens_without_stopwords)

print("Top 20 most frequent words:")
for word, count in combined_word_frequency.most_common(20):
    print(f"{word:<20} {count}")


Top 20 most frequent words:
language             4
natural              3
processing           3
data                 2
first                2
quality              2
enables              1
computers            1
analyze              1
human                1
useful               1
repeatable           1
ways                 1
acquisition          1
stage                1
reliable             1
pipeline             1
tokenization         1
divides              1
document             1


### 6(d) Print the combined document summary


In [28]:
total_headings = len(headings_a) + len(headings_b)
total_unique_body_paragraphs = len(deduplicated_paragraphs)
vocabulary_size = len(combined_word_frequency)
average_paragraph_length = round(
    sum(len(tokens_in_paragraph) for tokens_in_paragraph in paragraph_tokens)
    / len(paragraph_tokens),
    2,
) if paragraph_tokens else 0.0

if deduplicated_paragraphs:
    longest_index = max(
        range(len(deduplicated_paragraphs)),
        key=lambda index: len(paragraph_tokens[index]),
    )
    longest_combined_paragraph = deduplicated_paragraphs[longest_index]
    longest_word_count = len(paragraph_tokens[longest_index])
else:
    longest_combined_paragraph = ""
    longest_word_count = 0

comparison_summary = {
    "total_headings_across_both_documents": total_headings,
    "total_unique_body_paragraphs": total_unique_body_paragraphs,
    "vocabulary_size": vocabulary_size,
    "average_paragraph_length_words": average_paragraph_length,
    "longest_paragraph_word_count": longest_word_count,
    "longest_paragraph": longest_combined_paragraph,
}

pprint(comparison_summary)


{'average_paragraph_length_words': 12.25,
 'longest_paragraph': 'Natural language processing enables computers to '
                      'analyze human language in useful and repeatable ways.',
 'longest_paragraph_word_count': 14,
 'total_headings_across_both_documents': 6,
 'total_unique_body_paragraphs': 8,
 'vocabulary_size': 66}


## Task 7 Multi-Endpoint API Data Joining and Analysis

### 7(a) Fetch and validate users and posts


In [29]:
USERS_URL = "https://jsonplaceholder.typicode.com/users"
POSTS_URL = "https://jsonplaceholder.typicode.com/posts"

api_results = {}
for dataset_name, endpoint in {"users": USERS_URL, "posts": POSTS_URL}.items():
    response = requests.get(endpoint, timeout=30)
    if response.status_code != 200:
        raise ConnectionError(
            f"{dataset_name} endpoint returned status code {response.status_code}"
        )
    parsed_data = response.json()
    if not isinstance(parsed_data, list):
        raise TypeError(f"{dataset_name} endpoint did not return a JSON list")
    api_results[dataset_name] = parsed_data

users = api_results["users"]
posts = api_results["posts"]
print(f"Validated {len(users)} users and {len(posts)} posts.")


Validated 10 users and 100 posts.


### 7(b) Join users with their posts


In [30]:
posts_by_user_id = defaultdict(list)
for post in posts:
    posts_by_user_id[post["userId"]].append(post)

joined_users_posts = {}
for user in users:
    user_posts = posts_by_user_id[user["id"]]
    joined_users_posts[user["name"]] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": [post["title"] for post in user_posts],
    }

first_three_users = dict(list(joined_users_posts.items())[:3])
pprint(first_three_users)


{'Clementine Bauch': {'email': 'Nathan@yesenia.net',
                      'post_count': 10,
                      'titles': ['asperiores ea ipsam voluptatibus modi minima '
                                 'quia sint',
                                 'dolor sint quo a velit explicabo quia nam',
                                 'maxime id vitae nihil numquam',
                                 'autem hic labore sunt dolores incidunt',
                                 'rem alias distinctio quo quis',
                                 'est et quae odit qui non',
                                 'quasi id et eos tenetur aut quo autem',
                                 'delectus ullam et corporis nulla voluptas '
                                 'sequi',
                                 'iusto eius quod necessitatibus culpa ea',
                                 'a quo magni similique perferendis']},
 'Ervin Howell': {'email': 'Shanna@melissa.tv',
                  'post_count': 10,
        

### 7(c) Report the user with the most posts


In [31]:
most_active_user_name = max(
    joined_users_posts,
    key=lambda name: joined_users_posts[name]["post_count"],
)
most_active_summary = joined_users_posts[most_active_user_name]
most_active_user_record = next(
    user for user in users if user["name"] == most_active_user_name
)

print("Full name:", most_active_user_name)
print("Email address:", most_active_summary["email"])
print("Company name:", most_active_user_record["company"]["name"])
print("Post titles:")
for title_number, title in enumerate(most_active_summary["titles"], start=1):
    print(f"{title_number}. {title}")


Full name: Leanne Graham
Email address: Sincere@april.biz
Company name: Romaguera-Crona
Post titles:
1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
2. qui est esse
3. ea molestias quasi exercitationem repellat qui ipsa sit aut
4. eum et est occaecati
5. nesciunt quas odio
6. dolorem eum magni eos aperiam quia
7. magnam facilis autem
8. dolorem dolore est ipsam
9. nesciunt iure omnis dolorem tempora et accusantium
10. optio molestias id quia eum


### 7(d) Filter users by company description or city length


In [32]:
matching_users = []

for user in users:
    company_bs = user["company"]["bs"]
    city = user["address"]["city"]
    contains_synergies = re.search(r"\bsynergies\b", company_bs, flags=re.IGNORECASE)
    city_is_long = len(city) > 8

    if contains_synergies or city_is_long:
        matching_users.append(
            {
                "name": user["name"],
                "city": city,
                "post_count": len(posts_by_user_id[user["id"]]),
            }
        )

print("Matching users:")
for user in matching_users:
    print(
        f'{user["name"]} | City: {user["city"]} | '
        f'Posts: {user["post_count"]}'
    )


Matching users:
Leanne Graham | City: Gwenborough | Posts: 10
Ervin Howell | City: Wisokyburgh | Posts: 10
Clementine Bauch | City: McKenziehaven | Posts: 10
Patricia Lebsack | City: South Elvis | Posts: 10
Chelsey Dietrich | City: Roscoeview | Posts: 10
Mrs. Dennis Schulist | City: South Christy | Posts: 10
Kurtis Weissnat | City: Howemouth | Posts: 10
Nicholas Runolfsdottir V | City: Aliyaview | Posts: 10
Glenna Reichert | City: Bartholomebury | Posts: 10
Clementina DuBuque | City: Lebsackbury | Posts: 10


## Task 8 To-Do Completion Rate Analysis Across Users

### 8(a) Fetch and validate the to-do list


In [33]:
TODOS_URL = "https://jsonplaceholder.typicode.com/todos"

try:
    todos_response = requests.get(TODOS_URL, timeout=30)
    if todos_response.status_code != 200:
        raise ConnectionError(
            f"Todos endpoint returned status code {todos_response.status_code}"
        )

    todos = todos_response.json()
    if not isinstance(todos, list):
        raise TypeError("The parsed todos result is not a list.")
    if not todos:
        raise ValueError("The todos list contains no records.")

    print("Total todo records retrieved:", len(todos))
except (requests.RequestException, ValueError, TypeError, ConnectionError) as error:
    print("Unable to continue Task 8:", error)
    raise SystemExit(1)


Total todo records retrieved: 200


### 8(b) Group tasks by user and verify every total


In [34]:
todo_summary = defaultdict(lambda: {"total": 0, "completed": 0, "incomplete": 0})

for todo in todos:
    user_id = todo["userId"]
    todo_summary[user_id]["total"] += 1
    if todo["completed"]:
        todo_summary[user_id]["completed"] += 1
    else:
        todo_summary[user_id]["incomplete"] += 1

todo_summary = dict(sorted(todo_summary.items()))

for user_id, counts in todo_summary.items():
    total_is_valid = counts["completed"] + counts["incomplete"] == counts["total"]
    print(f"User {user_id}: {counts} | Valid total: {total_is_valid}")


User 1: {'total': 20, 'completed': 11, 'incomplete': 9} | Valid total: True
User 2: {'total': 20, 'completed': 8, 'incomplete': 12} | Valid total: True
User 3: {'total': 20, 'completed': 7, 'incomplete': 13} | Valid total: True
User 4: {'total': 20, 'completed': 6, 'incomplete': 14} | Valid total: True
User 5: {'total': 20, 'completed': 12, 'incomplete': 8} | Valid total: True
User 6: {'total': 20, 'completed': 6, 'incomplete': 14} | Valid total: True
User 7: {'total': 20, 'completed': 9, 'incomplete': 11} | Valid total: True
User 8: {'total': 20, 'completed': 11, 'incomplete': 9} | Valid total: True
User 9: {'total': 20, 'completed': 8, 'incomplete': 12} | Valid total: True
User 10: {'total': 20, 'completed': 12, 'incomplete': 8} | Valid total: True


### 8(c) Compute user and overall completion rates


In [35]:
completion_rates = {
    user_id: round(counts["completed"] / counts["total"] * 100, 1)
    for user_id, counts in todo_summary.items()
}

highest_rate_user = max(completion_rates, key=completion_rates.get)
lowest_rate_user = min(completion_rates, key=completion_rates.get)
overall_average_completion_rate = round(
    sum(completion_rates.values()) / len(completion_rates), 1
)

print(
    f"Highest completion rate: User {highest_rate_user} - "
    f"{completion_rates[highest_rate_user]:.1f}%"
)
print(
    f"Lowest completion rate: User {lowest_rate_user} - "
    f"{completion_rates[lowest_rate_user]:.1f}%"
)
print(f"Overall average completion rate: {overall_average_completion_rate:.1f}%")


Highest completion rate: User 5 - 60.0%
Lowest completion rate: User 4 - 30.0%
Overall average completion rate: 45.0%


### 8(d) Rank users in a formatted completion table


In [36]:
users_response = requests.get(USERS_URL, timeout=30)
if users_response.status_code != 200:
    raise ConnectionError(
        f"Users endpoint returned status code {users_response.status_code}"
    )

todo_users = users_response.json()
user_name_lookup = {user["id"]: user["name"] for user in todo_users}
ranked_user_ids = sorted(
    completion_rates,
    key=lambda user_id: (-completion_rates[user_id], user_id),
)

header = (
    f'{"Rank":<6} | {"Name":<25} | {"Total Tasks":<11} | '
    f'{"Completed":<9} | {"Completion Rate %":<17}'
)
print(header)
print("-" * len(header))

for rank, user_id in enumerate(ranked_user_ids, start=1):
    counts = todo_summary[user_id]
    print(
        f"{rank:<6} | {user_name_lookup[user_id]:<25} | "
        f'{counts["total"]:<11} | {counts["completed"]:<9} | '
        f"{completion_rates[user_id]:<17.1f}"
    )


Rank   | Name                      | Total Tasks | Completed | Completion Rate %
--------------------------------------------------------------------------------
1      | Chelsey Dietrich          | 20          | 12        | 60.0             
2      | Clementina DuBuque        | 20          | 12        | 60.0             
3      | Leanne Graham             | 20          | 11        | 55.0             
4      | Nicholas Runolfsdottir V  | 20          | 11        | 55.0             
5      | Kurtis Weissnat           | 20          | 9         | 45.0             
6      | Ervin Howell              | 20          | 8         | 40.0             
7      | Glenna Reichert           | 20          | 8         | 40.0             
8      | Clementine Bauch          | 20          | 7         | 35.0             
9      | Patricia Lebsack          | 20          | 6         | 30.0             
10     | Mrs. Dennis Schulist      | 20          | 6         | 30.0             


## Task 9 Nested JSON Flattening DataFrame Operations and Comment Analysis

### 9(a) Flatten every nested user record using exactly the required keys


In [37]:
flatten_users_response = requests.get(USERS_URL, timeout=30)
if flatten_users_response.status_code != 200:
    raise ConnectionError(
        f"Users endpoint returned status code {flatten_users_response.status_code}"
    )

nested_users = flatten_users_response.json()


def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }


flattened_users = [flatten_user(user) for user in nested_users]
pprint(flattened_users)


[{'city': 'Gwenborough',
  'company_catchphrase': 'Multi-layered client-server neural-net',
  'company_name': 'Romaguera-Crona',
  'email': 'Sincere@april.biz',
  'id': 1,
  'lat': '-37.3159',
  'lng': '81.1496',
  'name': 'Leanne Graham',
  'username': 'Bret',
  'zipcode': '92998-3874'},
 {'city': 'Wisokyburgh',
  'company_catchphrase': 'Proactive didactic contingency',
  'company_name': 'Deckow-Crist',
  'email': 'Shanna@melissa.tv',
  'id': 2,
  'lat': '-43.9509',
  'lng': '-34.4618',
  'name': 'Ervin Howell',
  'username': 'Antonette',
  'zipcode': '90566-7771'},
 {'city': 'McKenziehaven',
  'company_catchphrase': 'Face to face bifurcated interface',
  'company_name': 'Romaguera-Jacobson',
  'email': 'Nathan@yesenia.net',
  'id': 3,
  'lat': '-68.6102',
  'lng': '-47.0653',
  'name': 'Clementine Bauch',
  'username': 'Samantha',
  'zipcode': '59590-4157'},
 {'city': 'South Elvis',
  'company_catchphrase': 'Multi-tiered zero tolerance productivity',
  'company_name': 'Robel-Corkery'

### 9(b) Build the DataFrame and calculate distance from the origin


In [38]:
users_df = pd.DataFrame(flattened_users)
users_df["lat"] = users_df["lat"].astype(float)
users_df["lng"] = users_df["lng"].astype(float)
users_df["distance_from_origin"] = users_df.apply(
    lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2),
    axis=1,
)

print("Full flattened users DataFrame:")
print(users_df.to_string(index=False))

farthest_user = users_df.loc[users_df["distance_from_origin"].idxmax()]
print("\nUser with the greatest distance from origin:")
print(farthest_user.to_string())


Full flattened users DataFrame:
 id                     name         username                     email           city    zipcode      lat       lng       company_name                      company_catchphrase  distance_from_origin
  1            Leanne Graham             Bret         Sincere@april.biz    Gwenborough 92998-3874 -37.3159   81.1496    Romaguera-Crona   Multi-layered client-server neural-net             89.318161
  2             Ervin Howell        Antonette         Shanna@melissa.tv    Wisokyburgh 90566-7771 -43.9509  -34.4618       Deckow-Crist           Proactive didactic contingency             55.850669
  3         Clementine Bauch         Samantha        Nathan@yesenia.net  McKenziehaven 59590-4157 -68.6102  -47.0653 Romaguera-Jacobson        Face to face bifurcated interface             83.201575
  4         Patricia Lebsack         Karianne Julianne.OConner@kory.org    South Elvis 53919-4257  29.4572 -164.2990      Robel-Corkery Multi-tiered zero tolerance producti

### 9(c) Compare Northern and Southern Hemisphere users


In [39]:
northern_users = users_df.loc[users_df["lat"] > 0]
southern_users = users_df.loc[users_df["lat"] < 0]

print("Northern Hemisphere users:")
print(northern_users["name"].tolist())
print("\nSouthern Hemisphere users:")
print(southern_users["name"].tolist())

northern_average_distance = northern_users["distance_from_origin"].mean()
southern_average_distance = southern_users["distance_from_origin"].mean()

if northern_average_distance > southern_average_distance:
    higher_average_hemisphere = "Northern Hemisphere"
elif southern_average_distance > northern_average_distance:
    higher_average_hemisphere = "Southern Hemisphere"
else:
    higher_average_hemisphere = "Both hemispheres have the same average"

print(f"\nNorthern average distance: {northern_average_distance:.2f}")
print(f"Southern average distance: {southern_average_distance:.2f}")
print("Higher average distance:", higher_average_hemisphere)


Northern Hemisphere users:
['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']

Southern Hemisphere users:
['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']

Northern average distance: 123.58
Southern average distance: 84.32
Higher average distance: Northern Hemisphere


### 9(d) Count comments for every post and report the top and bottom five


In [40]:
COMMENTS_URL = "https://jsonplaceholder.typicode.com/comments"

posts_response = requests.get(POSTS_URL, timeout=30)
comments_response = requests.get(COMMENTS_URL, timeout=30)

for endpoint_name, response in {
    "posts": posts_response,
    "comments": comments_response,
}.items():
    if response.status_code != 200:
        raise ConnectionError(
            f"{endpoint_name} endpoint returned status code {response.status_code}"
        )

all_posts = posts_response.json()
all_comments = comments_response.json()
comment_counts = Counter(comment["postId"] for comment in all_comments)

post_comment_summary = sorted(
    [
        (post["id"], post["title"][:40], comment_counts[post["id"]])
        for post in all_posts
    ],
    key=lambda record: (-record[2], record[0]),
)

print("Top 5 most-commented posts:")
for record in post_comment_summary[:5]:
    print(record)

print("\nBottom 5 least-commented posts:")
for record in post_comment_summary[-5:]:
    print(record)


Top 5 most-commented posts:
(1, 'sunt aut facere repellat provident occae', 5)
(2, 'qui est esse', 5)
(3, 'ea molestias quasi exercitationem repell', 5)
(4, 'eum et est occaecati', 5)
(5, 'nesciunt quas odio', 5)

Bottom 5 least-commented posts:
(96, 'quaerat velit veniam amet cupiditate aut', 5)
(97, 'quas fugiat ut perspiciatis vero provide', 5)
(98, 'laboriosam dolor voluptates', 5)
(99, 'temporibus sit alias delectus eligendi p', 5)
(100, 'at nam consequatur ea labore ea harum', 5)


## Final Validation


In [41]:
required_page_keys = {"page_number", "raw_text", "word_count", "char_count"}
required_flat_user_keys = {
    "id", "name", "username", "email", "city", "zipcode", "lat", "lng",
    "company_name", "company_catchphrase",
}

assert len(page_texts) >= 5
assert all(set(page) == required_page_keys for page in pages_data)
assert len(paragraph_groups) == 3
assert first_table_df.shape == (3, 3)
assert all(
    counts["completed"] + counts["incomplete"] == counts["total"]
    for counts in todo_summary.values()
)
assert all(set(user) == required_flat_user_keys for user in flattened_users)
assert len(post_comment_summary) == len(all_posts)

print("All tasks executed successfully and all structural checks passed.")
print("Generated files:")
for generated_file in [PDF_SUMMARY_PATH, NLP_READY_PATH, DOCX_EXTRACTED_PATH]:
    print("-", generated_file)


All tasks executed successfully and all structural checks passed.
Generated files:
- lab02_work/pdf_summary.csv
- lab02_work/nlp_ready_output.txt
- lab02_work/docx_extracted.txt
